<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/ECO003b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulation parameters
N = 100
timesteps = 500
alpha_values = [0.1, 0.5, 1.0, 2.0, 5.0]
beta_values = [0.1, 0.5, 1.0, 2.0, 5.0]

# Entropy initialization functions
def gaussian_entropy(N):
    x = np.linspace(-1, 1, N)
    grid = np.meshgrid(x, x, x)
    r2 = sum(g**2 for g in grid)
    return np.exp(-5 * r2)

def uniform_entropy(N):
    return np.ones((N, N, N))

def bimodal_entropy(N):
    e = np.zeros((N, N, N))
    e[N//4,:,:] = 1
    e[3*N//4,:,:] = 1
    return e

def shell_entropy(N):
    x = np.linspace(-1, 1, N)
    grid = np.meshgrid(x, x, x)
    r = np.sqrt(sum(g**2 for g in grid))
    return np.exp(-((r - 0.5)**2) * 20)

def noisy_entropy(N):
    return np.random.rand(N, N, N)

# Collapse function with adaptive threshold
def simulate_collapse(entropy, alpha, beta):
    collapsed = np.zeros_like(entropy)
    for t in range(timesteps):
        gradient = np.gradient(entropy)
        magnitude = np.sqrt(sum(g**2 for g in gradient))
        non_zero_mag = magnitude[magnitude > 1e-9]
        if non_zero_mag.size == 0:
            adaptive_threshold = 0.0
        else:
            mu_grad = np.mean(non_zero_mag)
            sigma_grad = np.std(non_zero_mag)
            adaptive_threshold = alpha * mu_grad + beta * sigma_grad
        collapse_mask = magnitude > adaptive_threshold
        entropy[collapse_mask] *= 0.9
        collapsed += collapse_mask.astype(float)
        if np.sum(entropy) < 1e-6:
            break
    return collapsed

# Run sweep and record scores
distributions = {
    "Shell": shell_entropy(N),
    "Gaussian": gaussian_entropy(N),
    "Uniform": uniform_entropy(N),
    "Bimodal": bimodal_entropy(N),
    "Noisy": noisy_entropy(N)
}

score_table = []

for alpha in alpha_values:
    for beta in beta_values:
        scores = {}will
        for name, entropy in distributions.items():
            collapse_map = simulate_collapse(entropy.copy(), alpha, beta)
            scores[name] = np.sum(collapse_map)
        score = scores["Shell"] - (scores["Noisy"] + scores["Uniform"] + scores["Bimodal"])
        score_table.append((alpha, beta, scores["Shell"], score))

# Display results
print("α\tβ\tShell Fidelity\tScore (Shell - Noise)")
for row in score_table:
    print(f"{row[0]:.1f}\t{row[1]:.1f}\t{row[2]:.1f}\t\t{row[3]:.1f}")


α	β	Shell Fidelity	Score (Shell - Noise)
0.1	0.1	80457104.0		-233743376.0
0.1	0.5	48406704.0		-244790935.0
0.1	1.0	39865944.0		-231649047.0
0.1	2.0	29695768.0		-180720046.0
0.1	5.0	0.0		-22387724.0
0.5	0.1	82831672.0		-220178928.0
0.5	0.5	58359656.0		-222244171.0
0.5	1.0	35046800.0		-204394491.0
0.5	2.0	27369128.0		-89434115.0
0.5	5.0	0.0		-20767509.0
1.0	0.1	77538200.0		-144922644.0
1.0	0.5	54462144.0		-95350883.0
1.0	1.0	39025008.0		-26803347.0
1.0	2.0	18987136.0		10710664.0
1.0	5.0	0.0		-51556.0
2.0	0.1	54741928.0		52053834.0
2.0	0.5	36346264.0		34680481.0
2.0	1.0	31437728.0		30460443.0
2.0	2.0	0.0		-189413.0
2.0	5.0	0.0		0.0
5.0	0.1	0.0		0.0
5.0	0.5	0.0		0.0
5.0	1.0	0.0		0.0
5.0	2.0	0.0		0.0
5.0	5.0	0.0		0.0
